# Qwen-Image-Edit 2511 Uncensored **v1.1** — Colab runner
Text-to-image **and** image editing in one notebook.

**Before you start:** Runtime → Change runtime type → **GPU**.
⚠️ The v1.1 repo currently ships **FP8 safetensors (~29 GB total)**, not the small GGUF files its README mentions.
That needs a big GPU (**L4 24 GB or A100**). A free T4 will most likely run out of memory — on a T4 use the older v1 notebook (GGUF, 11.7 GB).

Run the 3 cells in order (cells 1 and 2 only once per session):
1. **Setup** – installs ComfyUI, looks at what the repo really contains, downloads the right files
2. **Start engine** – launches ComfyUI in the background and checks every file/node is found
3. **Generate** – leave `input_image_path` empty for text-to-image, or paste an image path to edit it

To get an image path: upload the image in the left sidebar (📁 Files), right-click it → **Copy path**.

In [ ]:
#@title 1️⃣ Setup: install ComfyUI + download model files (run once, ~10-20 min)
#@markdown "Auto" looks at the repo and downloads what really exists there (right now: FP8 files).
model_weights = "Auto (recommended)" #@param ["Auto (recommended)","FP8 safetensors (20 GB, needs 24GB+ GPU)","GGUF Q4_K_M (11.7 GB)","GGUF Q6_K (16.8 GB)","GGUF Q8_0 (21.7 GB)"]

import os, re, sys, json, shutil, subprocess, time

def sh(cmd, cwd=None, check=True):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError("Command failed: " + cmd)
    return r.returncode

PIP = sys.executable + " -m pip install -q "

# ---------- GPU / RAM check ----------
g = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                   shell=True, capture_output=True, text=True)
if g.returncode != 0 or not g.stdout.strip():
    raise RuntimeError("No GPU found. Runtime > Change runtime type > pick a GPU, then run this cell again.")
gpu_line = g.stdout.strip()
m = re.search(r"(\d+)\s*MiB", gpu_line)
gpu_mb = int(m.group(1)) if m else 0
ram_gb = int(open("/proc/meminfo").readline().split()[1]) / 1024 / 1024
print("GPU:", gpu_line, "| system RAM: %.1f GB" % ram_gb)

# ---------- ComfyUI ----------
if not os.path.exists("/content/ComfyUI/main.py"):
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI")
else:
    sh("git pull", cwd="/content/ComfyUI", check=False)
sh(PIP + "-r requirements.txt", cwd="/content/ComfyUI")

# ---------- look at what the repo REALLY contains ----------
from huggingface_hub import HfApi, hf_hub_download

REPO = "ChrisColeTech/qwen-image-edit-uncensored-v1.1-GGUF"
try:
    info = HfApi().model_info(REPO, files_metadata=True)
    sizes = {s.rfilename: (s.size or 0) for s in info.siblings}
except Exception as e:
    raise RuntimeError("Could not read the Hugging Face repo listing: " + repr(e)[:300])

def in_repo(prefix, suffix):
    return sorted(f for f in sizes if f.startswith(prefix) and f.endswith(suffix))

gguf_unets = in_repo("split/diffusion_models/", ".gguf")
st_unets   = in_repo("split/diffusion_models/", ".safetensors")
st_clips   = in_repo("split/text_encoders/", ".safetensors")
all_ggufs_te = in_repo("split/text_encoders/", ".gguf")
gguf_clips = [f for f in all_ggufs_te if "mmproj" not in f.lower()]
mmprojs    = [f for f in all_ggufs_te if "mmproj" in f.lower()]
vaes       = in_repo("split/vae/", ".safetensors")

print("\nFiles found in", REPO)
for f in sorted(sizes):
    if f.startswith("split/"):
        print("  %-75s %6.2f GB" % (f, sizes[f] / 1e9))

# ---------- choose the diffusion model ----------
def pick_gguf(q):
    hit = [f for f in gguf_unets if q in os.path.basename(f)]
    return hit[0] if hit else None

if model_weights.startswith("GGUF"):
    q = model_weights.split(" ")[1]
    unet_repo, unet_kind = pick_gguf(q), "gguf"
    if not unet_repo:
        raise RuntimeError("The v1.1 repo does NOT contain a GGUF file for %s. GGUF files present: %s. "
                           "Pick 'Auto' or 'FP8', or use the older v1 notebook for GGUF." % (q, gguf_unets or "none"))
elif model_weights.startswith("FP8"):
    if not st_unets:
        raise RuntimeError("No FP8 safetensors diffusion model found in the repo.")
    unet_repo, unet_kind = st_unets[0], "safetensors"
else:  # Auto
    if pick_gguf("Q4_K_M"):
        unet_repo, unet_kind = pick_gguf("Q4_K_M"), "gguf"
    elif gguf_unets:
        unet_repo, unet_kind = gguf_unets[0], "gguf"
    elif st_unets:
        unet_repo, unet_kind = st_unets[0], "safetensors"
    else:
        raise RuntimeError("No diffusion model (.gguf or .safetensors) found in the repo.")

# ---------- choose the text encoder ----------
if unet_kind == "gguf" and gguf_clips and mmprojs:
    clip_repos, clip_kind = [gguf_clips[0], mmprojs[0]], "gguf"
elif st_clips:
    clip_repos, clip_kind = [st_clips[0]], "safetensors"
elif gguf_clips and mmprojs:
    clip_repos, clip_kind = [gguf_clips[0], mmprojs[0]], "gguf"
else:
    raise RuntimeError("No text encoder found in the repo.")

if not vaes:
    raise RuntimeError("No VAE found in the repo.")
vae_repo = vaes[0]

plan = [(unet_repo, "diffusion_models")] + [(c, "text_encoders") for c in clip_repos] + [(vae_repo, "vae")]
need_gb = sum(sizes[f] for f, _ in plan) / 1e9
free_gb = shutil.disk_usage("/content").free / 1e9
print("\nPlan: diffusion=%s (%s) | text encoder=%s | download %.1f GB, free disk %.1f GB"
      % (os.path.basename(unet_repo), unet_kind, clip_kind, need_gb, free_gb))
if free_gb < need_gb + 3:
    raise RuntimeError("Not enough free disk space (%.1f GB free, %.1f GB needed)." % (free_gb, need_gb))

if unet_kind == "safetensors" and gpu_mb < 20000:
    print("\n" + "!" * 70)
    print("⚠️  WARNING: this model (FP8, ~29 GB in total) is too big for a %d MB GPU." % gpu_mb)
    print("    It will most likely crash with 'out of memory'. Use an L4 / A100 runtime,")
    print("    or use the older v1 GGUF notebook on a T4.")
    print("!" * 70 + "\n")

# ---------- GGUF loader (only if a GGUF file is used) ----------
if unet_kind == "gguf" or clip_kind == "gguf":
    for old in ["ComfyUI-GGUF", "comfyui-gguf"]:
        shutil.rmtree("/content/ComfyUI/custom_nodes/" + old, ignore_errors=True)
    cn = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF-Loader"
    if not os.path.exists(cn):
        sh("git clone --depth 1 https://github.com/ChrisColeTech/ComfyUI-GGUF-Loader " + cn)
    else:
        sh("git pull", cwd=cn, check=False)
    if os.path.exists(cn + "/requirements.txt"):
        sh(PIP + "-r requirements.txt", cwd=cn)
    sh(PIP + "--upgrade gguf")

# ---------- make sure the Python packages agree with each other ----------
def deps_ok():
    r = subprocess.run([sys.executable, "-c", "import transformers, huggingface_hub, PIL, requests"],
                       capture_output=True, text=True)
    return r.returncode == 0, r.stderr[-600:]

ok, err = deps_ok()
if not ok:
    print("Dependency conflict detected, repairing...\n", err)
    sh(PIP + '"huggingface_hub>=1.5.0,<2.0"', check=False)
    ok, err = deps_ok()
    if not ok:
        sh(PIP + "-U transformers", check=False)
        ok, err = deps_ok()
if not ok:
    raise RuntimeError("Package conflict could not be repaired:\n" + err)
print("Python packages OK ✅")

# ---------- download ----------
M = "/content/ComfyUI/models"

def fetch(repo_file, sub):
    name = os.path.basename(repo_file)
    dest_dir = os.path.join(M, sub)
    dest = os.path.join(dest_dir, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 0.98 * sizes[repo_file]:
        print("✔ already downloaded:", name)
        return name
    os.makedirs(dest_dir, exist_ok=True)
    for attempt in range(1, 4):
        try:
            print("⬇ downloading", name, "(%.1f GB, attempt %d)..." % (sizes[repo_file] / 1e9, attempt))
            p = hf_hub_download(repo_id=REPO, filename=repo_file, local_dir="/content/hf_dl")
            shutil.move(p, dest)
            print("  saved ->", dest)
            return name
        except Exception as e:
            print("  failed:", repr(e)[:300])
            time.sleep(5)
    raise RuntimeError("Could not download " + name + " after 3 tries. Run the cell again.")

unet_name = fetch(unet_repo, "diffusion_models")
clip_name = None
for i, c in enumerate(clip_repos):
    n = fetch(c, "text_encoders")
    if i == 0:
        clip_name = n
vae_name = fetch(vae_repo, "vae")

json.dump({"unet": unet_name, "unet_kind": unet_kind,
           "clip": clip_name, "clip_kind": clip_kind, "vae": vae_name},
          open("/content/qwen_edit_cfg.json", "w"))
print("\n✅ Setup finished. Now run cell 2.")


In [ ]:
#@title 2️⃣ Start the ComfyUI engine (run after cell 1, ~1-2 min)
low_vram_mode = False #@param {type:"boolean"}
#@markdown Turn on `low_vram_mode` only if you get "out of memory" errors.
restart_engine = False #@param {type:"boolean"}
#@markdown Tick `restart_engine` whenever you change the options in this cell.
extra_flags = "" #@param {type:"string"}
#@markdown Optional advanced flags, e.g. `--fp32-vae` (only if images come out black/garbled).

import os, sys, json, time, shlex, subprocess, requests

URL = "http://127.0.0.1:8188"

def alive():
    try:
        return requests.get(URL + "/system_stats", timeout=2).ok
    except Exception:
        return False

if restart_engine:
    subprocess.run("pkill -f 'main.py --listen'", shell=True)
    time.sleep(4)

if alive():
    print("Engine already running ✅")
else:
    os.makedirs("/content/outputs", exist_ok=True)
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--output-directory", "/content/outputs", "--disable-auto-launch"]
    if low_vram_mode:
        cmd.append("--lowvram")
    if extra_flags.strip():
        cmd += shlex.split(extra_flags)
    log = open("/content/comfyui.log", "w")
    subprocess.Popen(cmd, cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)
    print("Starting engine", end="")
    started = False
    for _ in range(150):
        if alive():
            started = True
            break
        print(".", end="", flush=True)
        time.sleep(2)
    if not started:
        print("\n❌ Engine did not start. Last log lines:\n")
        print("".join(open("/content/comfyui.log").readlines()[-40:]))
        raise RuntimeError("ComfyUI failed to start (copy the log above and send it to get it fixed)")
    print("\nEngine started ✅")

# ---------- verify everything the generator needs ----------
cfg = json.load(open("/content/qwen_edit_cfg.json"))
unet_node = "UnetLoaderGGUF" if cfg["unet_kind"] == "gguf" else "UNETLoader"
clip_node = "CLIPLoaderGGUF" if cfg["clip_kind"] == "gguf" else "CLIPLoader"

def node_info(name):
    return requests.get(URL + "/object_info/" + name).json().get(name)

def options(name, field):
    info = node_info(name)
    if not info:
        return None
    spec = info["input"]["required"][field]
    if isinstance(spec[0], list):
        return spec[0]
    if len(spec) > 1 and isinstance(spec[1], dict) and "options" in spec[1]:
        return spec[1]["options"]
    return []

problems = []
for n in [unet_node, clip_node, "TextEncodeQwenImageEdit", "CLIPTextEncode",
          "KSampler", "EmptyLatentImage", "VAELoader", "VAEDecode", "SaveImage", "LoadImage"]:
    if node_info(n):
        print("✅ node", n)
    else:
        print("❌ node missing:", n)
        problems.append(n)

for node, field, want in [(unet_node, "unet_name", cfg["unet"]),
                          (clip_node, "clip_name", cfg["clip"]),
                          ("VAELoader", "vae_name", cfg["vae"])]:
    opts = options(node, field)
    if opts is not None and want in opts:
        print("✅ file found:", want)
    else:
        print("❌ file NOT listed by", node, ":", want)
        problems.append(want)

types = options(clip_node, "type") or []
if "qwen_image" in types:
    print("✅ CLIP type qwen_image")
else:
    print("❌ CLIP type qwen_image not available in this ComfyUI build")
    problems.append("qwen_image type")

if problems:
    print("\nLast log lines:\n" + "".join(open("/content/comfyui.log").readlines()[-40:]))
    raise RuntimeError("Something is missing: " + ", ".join(problems) +
                       ". Run cell 1 again, then tick restart_engine and run this cell again.")
print("\n✅ Everything is ready. Now run cell 3.")


In [ ]:
#@title 3️⃣ Generate / Edit image
prompt = "put a pair of black sunglasses on the man" #@param {type:"string"}
input_image_path = "" #@param {type:"string"}
#@markdown **Empty = text-to-image.** Paste a path like `/content/photo.jpg` = edit that image with your prompt.
style = "None" #@param ["None","Natural","Cinematic","Photorealistic","3D Render","Animated / Cartoon","Anime","Digital Painting","Oil Painting","Watercolor","Pixel Art","Cyberpunk","Fantasy Art","Product Photo","Minimal Vector"]
#@markdown Style is added to the end of your prompt. Keep it `None` when editing an image.
aspect_ratio = "Match input image" #@param ["Match input image","1:1","16:9","9:16","4:3","3:4","3:2","2:3","21:9"]
megapixels = "1.0" #@param ["0.5","1.0","1.5","2.0"]
#@markdown 1.0 MP ≈ 1024×1024. On a free T4 stay at 0.5–1.0. "Match input image" only works when editing.
num_images = 1 #@param {type:"slider", min:1, max:8, step:1}
seed = -1 #@param {type:"integer"}
#@markdown seed = -1 means random.
#@markdown ---
#@markdown Advanced (this is a "turbo" model: 4 steps and cfg 1.0 are the recommended values, 8 steps = a bit better quality):
steps = 4 #@param {type:"slider", min:4, max:16, step:1}
cfg = 1.0 #@param {type:"number"}
negative_prompt = "" #@param {type:"string"}
#@markdown Negative prompt has no effect while cfg = 1.0.

import os, io, json, math, random, time, requests
from PIL import Image, ImageOps
from IPython.display import display

URL = "http://127.0.0.1:8188"
OUT_DIR = "/content/outputs"

STYLES = {
    "None": "",
    "Natural": "natural photograph, soft natural daylight, true-to-life colors, realistic textures, candid look, no heavy filters",
    "Cinematic": "cinematic film still, dramatic lighting, shallow depth of field, anamorphic lens, subtle film grain, color graded",
    "Photorealistic": "ultra realistic photograph, natural lighting, sharp focus, highly detailed, 85mm lens",
    "3D Render": "high quality 3D render, octane render, soft global illumination, subsurface scattering, detailed materials",
    "Animated / Cartoon": "stylized 3D animated movie look, expressive, vibrant colors, clean shapes, soft lighting",
    "Anime": "anime style illustration, clean line art, vibrant colors, detailed background, cel shading",
    "Digital Painting": "digital painting, concept art, painterly brush strokes, rich colors, highly detailed",
    "Oil Painting": "classical oil painting, visible brush strokes, rich texture, museum quality",
    "Watercolor": "watercolor illustration, soft washes, paper texture, delicate colors",
    "Pixel Art": "pixel art, 16-bit retro game style, limited color palette, crisp pixels",
    "Cyberpunk": "cyberpunk aesthetic, neon lights, rainy night, futuristic city, high contrast",
    "Fantasy Art": "epic fantasy art, magical atmosphere, intricate detail, volumetric light",
    "Product Photo": "professional product photography, studio lighting, clean background, sharp focus",
    "Minimal Vector": "flat vector illustration, minimal design, bold shapes, clean colors",
}

def snap(v, m=16, lo=256, hi=2048):
    return int(min(hi, max(lo, round(v / m) * m)))

def size_from_ratio(ratio, mp):
    a, b = [int(x) for x in ratio.split(":")]
    area = float(mp) * 1024 * 1024
    h = math.sqrt(area * b / a)
    return snap(h * a / b), snap(h)

def size_from_image(iw, ih, mp):
    area = float(mp) * 1024 * 1024
    h = math.sqrt(area * ih / iw)
    return snap(h * iw / ih), snap(h)

def engine_alive():
    try:
        return requests.get(URL + "/system_stats", timeout=3).ok
    except Exception:
        return False

def log_tail(n=30):
    try:
        return "".join(open("/content/comfyui.log").readlines()[-n:])
    except Exception:
        return "(no log)"

def upload_image(path):
    im = Image.open(path)
    im = ImageOps.exif_transpose(im).convert("RGB")
    if max(im.size) > 2048:
        im.thumbnail((2048, 2048), Image.LANCZOS)
    buf = io.BytesIO()
    im.save(buf, format="PNG")
    buf.seek(0)
    name = "colab_input_%d.png" % int(time.time())
    r = requests.post(URL + "/upload/image",
                      files={"image": (name, buf, "image/png")},
                      data={"type": "input", "overwrite": "true"})
    if r.status_code != 200:
        raise RuntimeError("Could not hand the image to ComfyUI: " + r.text[:300])
    j = r.json()
    value = j["name"] if not j.get("subfolder") else j["subfolder"] + "/" + j["name"]
    return value, im

def wait_for(pid, timeout=1800):
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            h = requests.get(URL + "/history/" + pid, timeout=10).json()
        except Exception:
            print("❌ The engine stopped responding (usually it ran out of RAM). Log tail:\n" + log_tail())
            raise RuntimeError("Engine crashed. Run cell 2 with restart_engine ticked and low_vram_mode on, "
                               "then lower megapixels to 0.5 and try again.")
        if pid in h:
            st = h[pid].get("status", {})
            if st.get("status_str") == "error":
                for kind, data in st.get("messages", []):
                    if kind == "execution_error":
                        print("❌", data.get("exception_type"), "-", data.get("exception_message"))
                raise RuntimeError("Generation failed (message above). If it says out of memory: cell 2 -> "
                                   "low_vram_mode on + restart_engine, and use megapixels 0.5.")
            if h[pid].get("outputs"):
                return h[pid]
        time.sleep(2)
    raise TimeoutError("Timed out waiting for the image.")

def side_by_side(a, b, height=512):
    a2 = a.resize((max(1, int(a.width * height / a.height)), height))
    b2 = b.resize((max(1, int(b.width * height / b.height)), height))
    canvas = Image.new("RGB", (a2.width + b2.width + 10, height), (30, 30, 30))
    canvas.paste(a2, (0, 0))
    canvas.paste(b2, (a2.width + 10, 0))
    return canvas

# ---------------- checks ----------------
if not prompt.strip():
    raise ValueError("Please type a prompt.")
if not engine_alive():
    raise RuntimeError("The engine is not running. Run cell 2 first.")
if not os.path.exists("/content/qwen_edit_cfg.json"):
    raise RuntimeError("Setup was not run. Run cell 1 first.")
cfgfile = json.load(open("/content/qwen_edit_cfg.json"))

path = input_image_path.strip().strip('"').strip("'")
edit_mode = bool(path)
if edit_mode:
    path = os.path.expanduser(path)
    if not os.path.isfile(path):
        folder = os.path.dirname(path) or "/content"
        print("❌ File not found:", path)
        if os.path.isdir(folder):
            print("Files in", folder, ":", [f for f in os.listdir(folder)][:30])
        raise FileNotFoundError("Right-click your uploaded image in the 📁 Files sidebar -> Copy path, and paste it into input_image_path.")

extra = STYLES[style]
full_prompt = prompt.strip() + (", " + extra if extra else "")

src_img, image_value = None, None
if edit_mode:
    image_value, src_img = upload_image(path)
    if aspect_ratio == "Match input image":
        W, H = size_from_image(src_img.width, src_img.height, megapixels)
    else:
        W, H = size_from_ratio(aspect_ratio, megapixels)
    mode_name = "IMAGE EDIT"
else:
    ratio = "1:1" if aspect_ratio == "Match input image" else aspect_ratio
    if aspect_ratio == "Match input image":
        print("ℹ️ No input image, so using 1:1.")
    W, H = size_from_ratio(ratio, megapixels)
    mode_name = "TEXT-TO-IMAGE"

print("Mode   :", mode_name)
print("Prompt :", full_prompt)
print("Size   : %dx%d | steps %d | cfg %s" % (W, H, steps, cfg))
print("(First image loads the model into memory — can take several minutes. Later ones are faster.)\n")

# ---------------- workflow graph ----------------
if cfgfile["unet_kind"] == "gguf":
    UNET_NODE = {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": cfgfile["unet"]}}
else:
    UNET_NODE = {"class_type": "UNETLoader", "inputs": {"unet_name": cfgfile["unet"], "weight_dtype": "default"}}
if cfgfile["clip_kind"] == "gguf":
    CLIP_NODE = {"class_type": "CLIPLoaderGGUF", "inputs": {"clip_name": cfgfile["clip"], "type": "qwen_image"}}
else:
    CLIP_NODE = {"class_type": "CLIPLoader", "inputs": {"clip_name": cfgfile["clip"], "type": "qwen_image"}}
enc_inputs = {"clip": ["2", 0], "prompt": full_prompt, "vae": ["3", 0]}
graph = {
    "1": UNET_NODE,
    "2": CLIP_NODE,
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": cfgfile["vae"]}},
    "4": {"class_type": "TextEncodeQwenImageEdit", "inputs": enc_inputs},
    "5": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["2", 0], "text": negative_prompt}},
    "6": {"class_type": "EmptyLatentImage", "inputs": {"width": W, "height": H, "batch_size": 1}},
    "7": {"class_type": "KSampler", "inputs": {
        "model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["6", 0],
        "seed": 0, "steps": int(steps), "cfg": float(cfg),
        "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0}},
    "8": {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
    "9": {"class_type": "SaveImage", "inputs": {"images": ["8", 0],
          "filename_prefix": "qwen_edit" if edit_mode else "qwen_t2i"}},
}
if edit_mode:
    graph["10"] = {"class_type": "LoadImage", "inputs": {"image": image_value}}
    enc_inputs["image"] = ["10", 0]

# ---------------- run ----------------
saved = []
for i in range(int(num_images)):
    s = (int(seed) + i) if int(seed) >= 0 else random.randint(0, 2**32 - 1)
    graph["7"]["inputs"]["seed"] = s
    t0 = time.time()
    r = requests.post(URL + "/prompt", json={"prompt": graph})
    if r.status_code != 200:
        print("❌ ComfyUI rejected the workflow:\n", r.text)
        raise RuntimeError("Workflow rejected — copy the message above and send it to get it fixed.")
    res = wait_for(r.json()["prompt_id"])
    for node_out in res["outputs"].values():
        for img in node_out.get("images", []):
            if img.get("type") == "temp":
                continue
            p = os.path.join(OUT_DIR, img.get("subfolder", ""), img["filename"])
            saved.append(p)
            print("✅ [%d/%d] seed %d | %.0fs | saved: %s" % (i + 1, int(num_images), s, time.time() - t0, p))
            out_im = Image.open(p).convert("RGB")
            if src_img is not None:
                print("   left = original, right = edited")
                display(side_by_side(src_img, out_im))
            else:
                pv = out_im.copy()
                pv.thumbnail((768, 768))
                display(pv)

print("\nAll images are in", OUT_DIR, "(left sidebar -> 📁 Files -> outputs)")
